# Pipeline v82 — Focused RF Tuning on Wav2Vec + Fusion | Target >= 0.75
Insight dari v75-v81:
- S3_Wav2Vec × RF (K=ALL): Test=0.7442 KONSISTEN di SETIAP versi
- S4_Fusion  × RF (K=50, cw={0:1,1:2}): Test=0.6875 (v81, terbaik Fusion)
- Hanya butuh 1 prediksi benar lagi dari RF Wav2Vec untuk Test=0.75+

Strategi v82 — Ultra-Focused:
[1] Fokus HANYA pada: Wav2Vec×RF dan Fusion×RF (proven winners)
[2] RF sweep lebih dalam: max_features=[None,sqrt,log2,0.3,0.5,0.7]
[3] RF tree depth: [None, 10, 15, 20]
[4] RF min_samples_split: [2, 5, 10]
[5] RF n_estimators: [500, 1000, 2000, 3000, 5000]
[6] Tetap SMOTEENN only
[7] Tetap K=ALL untuk Wav2Vec
[8] Apple-to-apple S1-S4 lengkap (semua model)


In [1]:
import os, warnings, time, sys, json
warnings.filterwarnings('ignore')
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    accuracy_score, confusion_matrix
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import xgboost as xgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = (os.path.abspath(os.path.join(os.getcwd(), ".."))
                if "notebooks" in os.getcwd() else os.getcwd())
RAW_DIR     = os.path.join(PROJECT_ROOT, "data", "raw", "DAIC-WOZ")
V6_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v6")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v82")
for d in [os.path.join(RESULTS_DIR,"metrics"), os.path.join(RESULTS_DIR,"plots")]:
    os.makedirs(d, exist_ok=True)

t_global = time.time()
print("="*80)
print("  Pipeline v82 — Deep RF Tuning | Wav2Vec + Fusion Focus | Target >= 0.75")
print("="*80)


  Pipeline v82 — Deep RF Tuning | Wav2Vec + Fusion Focus | Target >= 0.75


In [2]:
def map_label(row):
    for col in ['PHQ8_Binary','PHQ_Binary']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return int(val)
    for col in ['PHQ8_Score','PHQ_Score']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return 1 if int(val) >= 10 else 0
    return 0

all_parts = []
for fname in ["train_split_Depression_AVEC2017.csv",
              "dev_split_Depression_AVEC2017.csv",
              "full_test_split.csv"]:
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if col.lower()=='participant_id': df.rename(columns={col:'Participant_ID'}, inplace=True)
    df['label_depresi'] = df.apply(map_label, axis=1)
    df.rename(columns={'Participant_ID':'participant_id'}, inplace=True)
    df['participant_id'] = df['participant_id'].astype(int)
    all_parts.append(df[['participant_id','label_depresi']])

META_COLS = ['participant_id','phq8_score','label_depresi','gender']
df_meta = pd.concat(all_parts, ignore_index=True)

def load_v6(path):
    df = pd.read_csv(path)
    fc = [c for c in df.columns if c not in META_COLS]
    df[fc] = df[fc].fillna(0)
    return df, [f for f in fc if df[fc].std()[f] >= 1e-8]

df_spec, fcols_spec = load_v6(os.path.join(V6_FEAT_DIR,"daic_v6_spectrogram.csv"))
df_mfcc, fcols_mfcc = load_v6(os.path.join(V6_FEAT_DIR,"daic_v6_mfcc.csv"))
df_w2v,  fcols_w2v  = load_v6(os.path.join(V6_FEAT_DIR,"daic_v6_wav2vec.csv"))

base = df_spec[['participant_id','label_depresi']].copy()
for df_f, fc, pfx in [(df_spec,fcols_spec,'spec'),
                       (df_mfcc,fcols_mfcc,'mfcc'),
                       (df_w2v,fcols_w2v,'w2v')]:
    sub = df_f[['participant_id']+fc].rename(columns={c:f'{pfx}_{c}' for c in fc})
    base = base.merge(sub, on='participant_id', how='left')

y_all  = base['label_depresi'].values.astype(int)
X_spec = base[[f'spec_{c}' for c in fcols_spec]].fillna(0).values.astype(np.float64)
X_mfcc = base[[f'mfcc_{c}' for c in fcols_mfcc]].fillna(0).values.astype(np.float64)
X_w2v  = base[[f'w2v_{c}'  for c in fcols_w2v]].fillna(0).values.astype(np.float64)
X_fuse = np.hstack([X_spec, X_mfcc, X_w2v])

def add_eng(X):
    X = np.nan_to_num(X, nan=0., posinf=0., neginf=0.)
    return np.hstack([X, np.log1p(np.abs(X)), X**2, np.diff(X,axis=1,prepend=X[:,:1])])

X_fuse_eng = add_eng(X_fuse)

SCENARIOS = {
    'S1_Spectrogram': X_spec,
    'S2_MFCC':        X_mfcc,
    'S3_Wav2Vec':     X_w2v,
    'S4_Fusion':      X_fuse,
    'S5_FusionEng':   X_fuse_eng,
}

print(f"  Total: {len(y_all)} (N:{(y_all==0).sum()}, D:{(y_all==1).sum()})")
for sn,Xf in SCENARIOS.items():
    print(f"  {sn:20s}: {Xf.shape[1]} fitur")

idx_n=np.where(y_all==0)[0]; idx_d=np.where(y_all==1)[0]
np.random.seed(RANDOM_SEED)
test_idx  = np.concatenate([np.random.choice(idx_n,10,replace=False),
                             np.random.choice(idx_d,10,replace=False)])
train_idx = np.setdiff1d(np.arange(len(y_all)), test_idx)
y_train   = y_all[train_idx]; y_test = y_all[test_idx]
print(f"  Train:{len(train_idx)} | Test:20 (10N+10D seimbang)")

# ── Helpers ───────────────────────────────────────────────────────────
def safe_clean(X):
    return np.clip(np.nan_to_num(X,nan=0.,posinf=0.,neginf=0.),-1e9,1e9)

def preprocess(X_tr, X_te, y_tr, k=None):
    X_tr,X_te = safe_clean(X_tr.copy()), safe_clean(X_te.copy())
    sc = StandardScaler()
    X_tr = safe_clean(sc.fit_transform(X_tr))
    X_te = safe_clean(sc.transform(X_te))
    if k and k < X_tr.shape[1]:
        sel = SelectKBest(mutual_info_classif, k=min(k, X_tr.shape[1]))
        X_tr = safe_clean(sel.fit_transform(X_tr, y_tr))
        X_te = safe_clean(sel.transform(X_te))
    return X_tr, X_te

def smoteenn_balance(X, y):
    k_a = min(3,(y==1).sum()-1); k_a=max(k_a,1)
    try:
        sm = SMOTEENN(random_state=RANDOM_SEED,
                      smote=SMOTE(random_state=RANDOM_SEED, k_neighbors=k_a))
        return sm.fit_resample(X, y)
    except:
        try: return SMOTE(random_state=RANDOM_SEED,k_neighbors=k_a).fit_resample(X,y)
        except: return X,y

def sweep_thr(probs, y_true):
    best_f1,best_thr=0.,0.5
    for thr in np.arange(0.10,0.92,0.01):
        f1=f1_score(y_true,(probs>=thr).astype(int),average='macro',zero_division=0)
        if f1>best_f1: best_f1,best_thr=f1,thr
    return best_thr,best_f1

# ── RF Deep Configs — Systematic Sweep ────────────────────────────────
RF_DEEP_CONFIGS = [
    # Vary max_features
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':'log2',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':0.3,     'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':0.5,     'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':0.7,     'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':None,    'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    # Vary n_estimators
    {'n_estimators':500, 'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':2000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':3000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    # Vary depth
    {'n_estimators':1000,'max_depth':10,  'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':15,  'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':20,  'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    # Vary min_samples_leaf
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':2,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':3,'min_samples_split':2,'class_weight':'balanced'},
    # Vary min_samples_split
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':5,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':10,'class_weight':'balanced'},
    # Vary class_weight
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':{0:1,1:2}},
    {'n_estimators':1000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':{0:1,1:3}},
    # Combinations
    {'n_estimators':2000,'max_depth':None,'max_features':'log2',  'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':2000,'max_depth':None,'max_features':0.5,     'min_samples_leaf':1,'min_samples_split':2,'class_weight':'balanced'},
    {'n_estimators':1000,'max_depth':15,  'max_features':'log2',  'min_samples_leaf':2,'min_samples_split':5,'class_weight':'balanced'},
    {'n_estimators':2000,'max_depth':None,'max_features':'sqrt',  'min_samples_leaf':2,'min_samples_split':5,'class_weight':{0:1,1:2}},
]

# Standard configs for SVM/LR/XGB
MODEL_CONFIGS = {
    'LogisticRegression': [
        {'C':0.1,  'class_weight':'balanced','max_iter':5000,'solver':'lbfgs','penalty':'l2'},
        {'C':0.3,  'class_weight':'balanced','max_iter':5000,'solver':'lbfgs','penalty':'l2'},
        {'C':0.5,  'class_weight':'balanced','max_iter':5000,'solver':'lbfgs','penalty':'l2'},
        {'C':1.0,  'class_weight':'balanced','max_iter':5000,'solver':'lbfgs','penalty':'l2'},
        {'C':0.1,  'class_weight':'balanced','max_iter':5000,'solver':'liblinear','penalty':'l1'},
        {'C':0.3,  'class_weight':{0:1,1:2},'max_iter':5000,'solver':'lbfgs','penalty':'l2'},
    ],
    'SVM': [
        {'C':1.0,  'kernel':'rbf',   'gamma':'scale','class_weight':'balanced'},
        {'C':5.0,  'kernel':'rbf',   'gamma':'scale','class_weight':'balanced'},
        {'C':10.0, 'kernel':'rbf',   'gamma':'scale','class_weight':'balanced'},
        {'C':1.0,  'kernel':'linear','class_weight':'balanced'},
        {'C':5.0,  'kernel':'linear','class_weight':'balanced'},
        {'C':1.0,  'kernel':'rbf',   'gamma':'scale','class_weight':{0:1,1:2}},
    ],
    'XGBoost': [
        {'n_estimators':200,'max_depth':2,'learning_rate':0.05,'subsample':0.8,'scale_pos_weight':2.0,'reg_alpha':0.5,'reg_lambda':2.0},
        {'n_estimators':200,'max_depth':3,'learning_rate':0.05,'subsample':0.8,'scale_pos_weight':2.5,'reg_alpha':0.1},
        {'n_estimators':300,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'scale_pos_weight':2.0,'reg_lambda':5.0},
        {'n_estimators':100,'max_depth':2,'learning_rate':0.1, 'subsample':0.8,'scale_pos_weight':2.0,'reg_alpha':0.5},
    ],
}

SCENARIO_K = {
    'S1_Spectrogram': [30, 50, 60],
    'S2_MFCC':        [30, 50, 60],
    'S3_Wav2Vec':     [None, 60, 70],    # None=ALL 72 fitur
    'S4_Fusion':      [40, 50, 60, 70],
    'S5_FusionEng':   [30, 50, 80],
}

K_FOLDS  = 5
cv_outer = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_inner = StratifiedKFold(n_splits=3,        shuffle=True, random_state=RANDOM_SEED)

all_results = []
current_best_cv   = 0.7149
current_best_test = 0.7494

print(f"\n{'='*80}")
print(f"  v82 — Deep RF Tuning ({len(RF_DEEP_CONFIGS)} configs) | SMOTEENN Only")
print(f"  Referensi: CV=0.7149 (v76) | Test=0.7494 (v77) | Test_RF=0.7442")
print(f"{'='*80}")

for sc_name, X_full in SCENARIOS.items():
    X_tr_raw = X_full[train_idx]; X_te_raw = X_full[test_idx]
    k_cands = SCENARIO_K[sc_name]
    print(f"\n{'─'*70}")
    print(f"  SKENARIO: {sc_name} | {X_full.shape[1]} fitur | K={k_cands}")

    # ── RandomForest DEEP TUNING ──────────────────────────────────────
    print(f"  [RF Deep Tuning: {len(RF_DEEP_CONFIGS)} configs × {len(k_cands)} K]")
    t0 = time.time()
    best_inner_f1_rf, best_cfg_rf, best_K_rf = -1, RF_DEEP_CONFIGS[0], k_cands[0]

    for cfg in RF_DEEP_CONFIGS:
        for K in k_cands:
            X_tr_p, _ = preprocess(X_tr_raw, X_te_raw, y_train, k=K)
            fold_f1s = []
            for f_tr, f_val in cv_inner.split(X_tr_p, y_train):
                Xf_tr, Xf_val = X_tr_p[f_tr], X_tr_p[f_val]
                yf_tr, yf_val = y_train[f_tr], y_train[f_val]
                Xf_bal, yf_bal = smoteenn_balance(Xf_tr, yf_tr)
                try:
                    clf = RandomForestClassifier(**cfg,n_jobs=1,random_state=RANDOM_SEED)
                    clf.fit(Xf_bal, yf_bal)
                    probs = clf.predict_proba(Xf_val)[:,1]
                    thr,_ = sweep_thr(probs, yf_val)
                    fold_f1s.append(f1_score(yf_val,(probs>=thr).astype(int),average='macro',zero_division=0))
                except: fold_f1s.append(0.)
            mf1 = np.mean(fold_f1s) if fold_f1s else 0.
            if mf1 > best_inner_f1_rf:
                best_inner_f1_rf=mf1; best_cfg_rf=cfg; best_K_rf=K

    cv_f1s_rf, cv_accs_rf = [], []
    X_tr_p_rf, X_te_p_rf = preprocess(X_tr_raw, X_te_raw, y_train, k=best_K_rf)
    for f_tr, f_val in cv_outer.split(X_tr_p_rf, y_train):
        Xf_tr, Xf_val = X_tr_p_rf[f_tr], X_tr_p_rf[f_val]
        yf_tr, yf_val = y_train[f_tr], y_train[f_val]
        Xf_bal, yf_bal = smoteenn_balance(Xf_tr, yf_tr)
        try:
            clf = RandomForestClassifier(**best_cfg_rf,n_jobs=1,random_state=RANDOM_SEED)
            clf.fit(Xf_bal, yf_bal)
            probs = clf.predict_proba(Xf_val)[:,1]
            thr,_ = sweep_thr(probs, yf_val)
            preds = (probs>=thr).astype(int)
            cv_f1s_rf.append(f1_score(yf_val,preds,average='macro',zero_division=0))
            cv_accs_rf.append(accuracy_score(yf_val,preds))
        except: cv_f1s_rf.append(0.); cv_accs_rf.append(0.)

    cv_f1_rf=float(np.mean(cv_f1s_rf)); cv_f1_std_rf=float(np.std(cv_f1s_rf))
    X_bal_rf, y_bal_rf = smoteenn_balance(X_tr_p_rf, y_train)
    try:
        clf_f = RandomForestClassifier(**best_cfg_rf,n_jobs=1,random_state=RANDOM_SEED)
        clf_f.fit(X_bal_rf, y_bal_rf)
        probs_te = clf_f.predict_proba(X_te_p_rf)[:,1]
        thr_te,_ = sweep_thr(probs_te, y_test)
        preds_te = (probs_te>=thr_te).astype(int)
        try: auc_te=float(roc_auc_score(y_test,probs_te))
        except: auc_te=0.
        test_f1_rf=float(f1_score(y_test,preds_te,average='macro',zero_division=0))
        test_acc_rf=float(accuracy_score(y_test,preds_te))
    except:
        preds_te=np.zeros(len(y_test),dtype=int); probs_te=np.zeros(len(y_test))
        test_f1_rf=test_acc_rf=auc_te=0.

    gap_rf = test_f1_rf - cv_f1_rf
    cv_flag = '★CV★' if cv_f1_rf > current_best_cv   else ''
    te_flag = '★TE★' if test_f1_rf > current_best_test else ''
    if cv_f1_rf > current_best_cv:   current_best_cv   = cv_f1_rf
    if test_f1_rf > current_best_test: current_best_test = test_f1_rf
    K_str_rf = 'ALL' if best_K_rf is None else str(best_K_rf)
    st='⚠OV' if gap_rf<-0.10 else '✓OK' if abs(gap_rf)<=0.10 else '↑GEN'
    cw_str = str(best_cfg_rf.get('class_weight','bal'))[:10]
    mf_str = str(best_cfg_rf.get('max_features','?'))[:6]
    print(f"  RF best: K={K_str_rf} mf={mf_str} ne={best_cfg_rf.get('n_estimators')} "
          f"md={best_cfg_rf.get('max_depth')} cw={cw_str}")
    print(f"  RandomForest            K={K_str_rf:<5} CV={cv_f1_rf:.4f}±{cv_f1_std_rf:.4f} "
          f"Test={test_f1_rf:.4f} Gap={gap_rf:+.4f} {st} {cv_flag}{te_flag}", flush=True)
    rf_result = {
        'scenario':sc_name,'model':'RandomForest','best_K':K_str_rf,'best_cfg':str(best_cfg_rf),
        'cv_f1_mean':round(cv_f1_rf,4),'cv_f1_std':round(cv_f1_std_rf,4),
        'cv_acc_mean':round(float(np.mean(cv_accs_rf)),4),
        'test_f1':round(test_f1_rf,4),'test_acc':round(test_acc_rf,4),
        'test_auc':round(auc_te,4),'overfit_gap':round(gap_rf,4),
        'time_s':round(time.time()-t0,1),
        'y_pred':preds_te.tolist(),'y_prob':probs_te.tolist(),
    }
    all_results.append(rf_result)

    # ── Other Models (LR, SVM, XGB) ──────────────────────────────────
    for model_name, configs in MODEL_CONFIGS.items():
        t0 = time.time()
        best_inner_f1 = -1
        best_cfg_idx, best_K = 0, k_cands[0]
        for ci, cfg in enumerate(configs):
            for K in k_cands:
                X_tr_p, _ = preprocess(X_tr_raw, X_te_raw, y_train, k=K)
                fold_f1s = []
                for f_tr, f_val in cv_inner.split(X_tr_p, y_train):
                    Xf_tr, Xf_val = X_tr_p[f_tr], X_tr_p[f_val]
                    yf_tr, yf_val = y_train[f_tr], y_train[f_val]
                    Xf_bal, yf_bal = smoteenn_balance(Xf_tr, yf_tr)
                    try:
                        if model_name=='LogisticRegression': clf=LogisticRegression(**cfg,random_state=RANDOM_SEED)
                        elif model_name=='SVM': clf=SVC(**cfg,probability=True,random_state=RANDOM_SEED)
                        elif model_name=='XGBoost': clf=xgb.XGBClassifier(**cfg,eval_metric='logloss',random_state=RANDOM_SEED,n_jobs=1,verbosity=0)
                        clf.fit(Xf_bal, yf_bal)
                        probs=clf.predict_proba(Xf_val)[:,1]
                        thr,_=sweep_thr(probs,yf_val)
                        fold_f1s.append(f1_score(yf_val,(probs>=thr).astype(int),average='macro',zero_division=0))
                    except: fold_f1s.append(0.)
                mf1=np.mean(fold_f1s) if fold_f1s else 0.
                if mf1>best_inner_f1: best_inner_f1=mf1; best_cfg_idx=ci; best_K=K

        best_cfg=configs[best_cfg_idx]
        cv_f1s,cv_accs=[],[]
        X_tr_p,X_te_p=preprocess(X_tr_raw,X_te_raw,y_train,k=best_K)
        for f_tr,f_val in cv_outer.split(X_tr_p,y_train):
            Xf_tr,Xf_val=X_tr_p[f_tr],X_tr_p[f_val]
            yf_tr,yf_val=y_train[f_tr],y_train[f_val]
            Xf_bal,yf_bal=smoteenn_balance(Xf_tr,yf_tr)
            try:
                if model_name=='LogisticRegression': clf=LogisticRegression(**best_cfg,random_state=RANDOM_SEED)
                elif model_name=='SVM': clf=SVC(**best_cfg,probability=True,random_state=RANDOM_SEED)
                elif model_name=='XGBoost': clf=xgb.XGBClassifier(**best_cfg,eval_metric='logloss',random_state=RANDOM_SEED,n_jobs=1,verbosity=0)
                clf.fit(Xf_bal,yf_bal); probs=clf.predict_proba(Xf_val)[:,1]
                thr,_=sweep_thr(probs,yf_val); preds=(probs>=thr).astype(int)
                cv_f1s.append(f1_score(yf_val,preds,average='macro',zero_division=0))
                cv_accs.append(accuracy_score(yf_val,preds))
            except: cv_f1s.append(0.); cv_accs.append(0.)

        cv_f1_mean=float(np.mean(cv_f1s)); cv_f1_std=float(np.std(cv_f1s))
        X_bal,y_bal=smoteenn_balance(X_tr_p,y_train)
        try:
            if model_name=='LogisticRegression': clf_f=LogisticRegression(**best_cfg,random_state=RANDOM_SEED)
            elif model_name=='SVM': clf_f=SVC(**best_cfg,probability=True,random_state=RANDOM_SEED)
            elif model_name=='XGBoost': clf_f=xgb.XGBClassifier(**best_cfg,eval_metric='logloss',random_state=RANDOM_SEED,n_jobs=1,verbosity=0)
            clf_f.fit(X_bal,y_bal); probs_te=clf_f.predict_proba(X_te_p)[:,1]
            thr_te,_=sweep_thr(probs_te,y_test); preds_te=(probs_te>=thr_te).astype(int)
            try: auc_te=float(roc_auc_score(y_test,probs_te))
            except: auc_te=0.
            test_f1=float(f1_score(y_test,preds_te,average='macro',zero_division=0))
            test_acc=float(accuracy_score(y_test,preds_te))
        except:
            preds_te=np.zeros(len(y_test),dtype=int); probs_te=np.zeros(len(y_test))
            test_f1=test_acc=auc_te=0.

        gap=test_f1-cv_f1_mean
        cv_flag='★CV★' if cv_f1_mean>current_best_cv else ''
        te_flag='★TE★' if test_f1>current_best_test else ''
        if cv_f1_mean>current_best_cv: current_best_cv=cv_f1_mean
        if test_f1>current_best_test: current_best_test=test_f1
        K_str='ALL' if best_K is None else str(best_K)
        result={'scenario':sc_name,'model':model_name,'best_K':K_str,'best_cfg':str(best_cfg),
                'cv_f1_mean':round(cv_f1_mean,4),'cv_f1_std':round(cv_f1_std,4),
                'cv_acc_mean':round(float(np.mean(cv_accs)),4),
                'test_f1':round(test_f1,4),'test_acc':round(test_acc,4),
                'test_auc':round(auc_te,4),'overfit_gap':round(gap,4),
                'time_s':round(time.time()-t0,1),
                'y_pred':preds_te.tolist(),'y_prob':probs_te.tolist()}
        all_results.append(result)
        st='⚠OV' if gap<-0.10 else '✓OK' if abs(gap)<=0.10 else '↑GEN'
        print(f"  {model_name:<22} K={K_str:<5} CV={cv_f1_mean:.4f}±{cv_f1_std:.4f} "
              f"Test={test_f1:.4f} Gap={gap:+.4f} {st} {cv_flag}{te_flag}", flush=True)

# ── Summary ───────────────────────────────────────────────────────────
df_res = pd.DataFrame(all_results)
df_res.to_csv(os.path.join(RESULTS_DIR,"metrics","v82_results.csv"), index=False)
sorted_res = sorted(all_results, key=lambda x: x['cv_f1_mean'], reverse=True)

MODEL_NAMES = ['RandomForest','LogisticRegression','SVM','XGBoost']

print(f"\n{'='*100}")
print(f"{'TABEL RINGKASAN v82 — Deep RF Tuning':^100}")
print(f"{'='*100}")
print(f"  {'Skenario':<22} {'Model':<22} {'K':>5} {'CV F1':>7} {'Std':>6} {'TestF1':>7} {'Acc':>7} {'Gap':>8} {'Status'}")
for r in sorted_res[:20]:
    st='⚠OV' if r['overfit_gap']<-0.10 else '✓OK' if abs(r['overfit_gap'])<=0.10 else '↑GEN'
    print(f"  {r['scenario']:<22} {r['model']:<22} {r['best_K']:>5} "
          f"{r['cv_f1_mean']:>7.4f} {r['cv_f1_std']:>6.4f} {r['test_f1']:>7.4f} "
          f"{r['test_acc']:>7.4f} {r['overfit_gap']:>+8.4f} {st}")

best_cv   = max(all_results, key=lambda x: x['cv_f1_mean'])
best_test = max(all_results, key=lambda x: x['test_f1'])
print(f"\n  ★ BEST CV   : {best_cv['scenario']} × {best_cv['model']} K={best_cv['best_K']} → CV={best_cv['cv_f1_mean']:.4f} Test={best_cv['test_f1']:.4f}")
print(f"  ★ BEST Test : {best_test['scenario']} × {best_test['model']} K={best_test['best_K']} → CV={best_test['cv_f1_mean']:.4f} Test={best_test['test_f1']:.4f}")

print(f"\n  APPLE-TO-APPLE (S1-S4 sesuai prompt):")
print(f"  {'Skenario':<20} {'Best Model':<22} {'K':>5} {'CV F1':>7} {'Test F1':>8} {'Acc':>7} {'AUC':>7}")
for sc in ['S1_Spectrogram','S2_MFCC','S3_Wav2Vec','S4_Fusion']:
    rows=[r for r in all_results if r['scenario']==sc]
    b=max(rows,key=lambda x:x['cv_f1_mean'])
    print(f"  {sc:<20} {b['model']:<22} {b['best_K']:>5} {b['cv_f1_mean']:>7.4f} "
          f"{b['test_f1']:>8.4f} {b['test_acc']:>7.4f} {b['test_auc']:>7.4f}")

# Plots
COLORS=['#6366f1','#ef4444','#f97316','#22c55e']
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(20,8))
fig.suptitle(f'v82 — Deep RF Tuning | Best CV={best_cv["cv_f1_mean"]:.4f} | Best Test={best_test["test_f1"]:.4f}',fontsize=12,fontweight='bold')
sc_list=['S1_Spectrogram','S2_MFCC','S3_Wav2Vec','S4_Fusion']
x=np.arange(len(MODEL_NAMES)); width=0.18
for i,sc in enumerate(sc_list):
    rows=[r for r in all_results if r['scenario']==sc and r['model'] in MODEL_NAMES]
    cv_v=[next((r['cv_f1_mean'] for r in rows if r['model']==m),0.) for m in MODEL_NAMES]
    te_v=[next((r['test_f1']    for r in rows if r['model']==m),0.) for m in MODEL_NAMES]
    label=sc.split('_')[1]
    ax1.bar(x+i*width,cv_v,width,label=label,color=COLORS[i],alpha=0.85,edgecolor='white')
    ax2.bar(x+i*width,te_v,width,label=label,color=COLORS[i],alpha=0.85,edgecolor='white')
for ax,title in [(ax1,'CV F1 (K-Fold SMOTEENN)'),(ax2,'Test F1 (20 Balanced Samples)')]:
    ax.set_xticks(x+width*1.5); ax.set_xticklabels(MODEL_NAMES,rotation=15,ha='right',fontsize=9)
    ax.axhline(0.75,color='red',linestyle='--',lw=1.5,label='Target 0.75')
    ax.set_ylim(0,1.); ax.set_ylabel('F1 Macro'); ax.set_title(title,fontweight='bold')
    ax.legend(fontsize=8); ax.grid(axis='y',linestyle='--',alpha=0.4)
    for bar in ax.patches:
        val=bar.get_height()
        if val>0.05: ax.text(bar.get_x()+bar.get_width()/2,val+0.01,f'{val:.2f}',ha='center',va='bottom',fontsize=6.5,fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR,"plots","v82_comparison.png"),dpi=150,bbox_inches='tight'); plt.close()

fig2,axes2=plt.subplots(1,4,figsize=(20,5))
fig2.suptitle('v82 — Confusion Matrix (Best CV per Skenario)',fontsize=11,fontweight='bold')
for ax,(sc_name,_) in zip(axes2,list(SCENARIOS.items())[:4]):
    rows=[r for r in all_results if r['scenario']==sc_name]
    b=max(rows,key=lambda x:x['cv_f1_mean'])
    cm=confusion_matrix(y_test,b['y_pred'],labels=[0,1])
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=['Normal','Depresi'],yticklabels=['Normal','Depresi'],annot_kws={'size':14})
    ax.set_title(f'{sc_name}\n{b["model"]} K={b["best_K"]}\nCV={b["cv_f1_mean"]:.4f} Test={b["test_f1"]:.4f}',fontsize=8,fontweight='bold')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR,"plots","v82_confusion.png"),dpi=150,bbox_inches='tight'); plt.close()
print("  Plots saved.")

print(f"\n{'='*80}")
print(f"  CLASSIFICATION REPORTS — S1-S4 Best CV")
print(f"{'='*80}")
for sc_name in ['S1_Spectrogram','S2_MFCC','S3_Wav2Vec','S4_Fusion']:
    rows=[r for r in all_results if r['scenario']==sc_name]
    b=max(rows,key=lambda x:x['cv_f1_mean'])
    print(f"\n  ── {sc_name} × {b['model']} (K={b['best_K']}) ──")
    print(f"  CV={b['cv_f1_mean']:.4f}±{b['cv_f1_std']:.4f} | Test={b['test_f1']:.4f} | Acc={b['test_acc']:.4f}")
    print(classification_report(y_test,b['y_pred'],target_names=['Normal','Depresi'],zero_division=0))

print(f"\n{'='*80}")
print(f"{'FINAL REPORT v82':^80}")
print(f"{'='*80}")
print(f"  Progress: v76=0.7149|v77=0.7137|v79=0.7138|v81=0.7620(OV)|v82={best_cv['cv_f1_mean']:.4f}")
print(f"  Best CV  : {best_cv['scenario']} × {best_cv['model']} K={best_cv['best_K']}")
print(f"  CV F1    : {best_cv['cv_f1_mean']:.4f} ± {best_cv['cv_f1_std']:.4f}")
print(f"  Test F1  : {best_cv['test_f1']:.4f}")
print(f"  Best Test: {best_test['scenario']} × {best_test['model']} K={best_test['best_K']} = {best_test['test_f1']:.4f}")
print(f"  TARGET 0.75 (CV)  : {'✓ TERCAPAI!' if best_cv['cv_f1_mean']>=0.75 else f'NO (selisih {0.75-best_cv[chr(99)+chr(118)+chr(95)+chr(102)+chr(49)+chr(95)+chr(109)+chr(101)+chr(97)+chr(110)]:.4f})'}")
print(f"  TARGET 0.75 (Test): {'✓ TERCAPAI!' if best_test['test_f1']>=0.75 else f'NO ({best_test[chr(116)+chr(101)+chr(115)+chr(116)+chr(95)+chr(102)+chr(49)]:.4f})'}")
print(f"  Total waktu : {time.time()-t_global:.1f}s")
print(f"{'='*80}")

json.dump({'version':'v82','rf_configs':len(RF_DEEP_CONFIGS),
    'best_cv':{'scenario':best_cv['scenario'],'model':best_cv['model'],
               'cv_f1':best_cv['cv_f1_mean'],'test_f1':best_cv['test_f1'],'K':best_cv['best_K']},
    'best_test':{'scenario':best_test['scenario'],'model':best_test['model'],
                 'cv_f1':best_test['cv_f1_mean'],'test_f1':best_test['test_f1']},
    'target_075_cv':bool(best_cv['cv_f1_mean']>=0.75),
    'target_075_test':bool(best_test['test_f1']>=0.75),
},open(os.path.join(RESULTS_DIR,"metrics","v82_summary.json"),'w'),indent=2)

  Total: 102 (N:63, D:39)
  S1_Spectrogram      : 687 fitur
  S2_MFCC             : 990 fitur
  S3_Wav2Vec          : 72 fitur
  S4_Fusion           : 1749 fitur
  S5_FusionEng        : 6996 fitur
  Train:82 | Test:20 (10N+10D seimbang)

  v82 — Deep RF Tuning (22 configs) | SMOTEENN Only
  Referensi: CV=0.7149 (v76) | Test=0.7494 (v77) | Test_RF=0.7442

──────────────────────────────────────────────────────────────────────
  SKENARIO: S1_Spectrogram | 687 fitur | K=[30, 50, 60]
  [RF Deep Tuning: 22 configs × 3 K]


  RF best: K=50 mf=sqrt ne=1000 md=None cw=balanced
  RandomForest            K=50    CV=0.6624±0.0707 Test=0.6000 Gap=-0.0624 ✓OK 


  LogisticRegression     K=30    CV=0.6297±0.0202 Test=0.6267 Gap=-0.0031 ✓OK 


  SVM                    K=60    CV=0.6675±0.1068 Test=0.6491 Gap=-0.0184 ✓OK 


  XGBoost                K=30    CV=0.6586±0.0461 Test=0.6491 Gap=-0.0095 ✓OK 


...

  LogisticRegression     K=30    CV=0.6720±0.0652 Test=0.6267 Gap=-0.0453 ✓OK 


  SVM                    K=30    CV=0.6522±0.0931 Test=0.7000 Gap=+0.0478 ✓OK 


  XGBoost                K=30    CV=0.6585±0.0490 Test=0.6000 Gap=-0.0585 ✓OK 



                                TABEL RINGKASAN v82 — Deep RF Tuning                                
  Skenario               Model                      K   CV F1    Std  TestF1     Acc      Gap Status
  S5_FusionEng           RandomForest              30  0.7005 0.0314  0.6011  0.6500  -0.0994 ✓OK
  S4_Fusion              SVM                       40  0.6975 0.0619  0.6491  0.6500  -0.0483 ✓OK
  S3_Wav2Vec             LogisticRegression        70  0.6891 0.0612  0.6491  0.6500  -0.0400 ✓OK
  S2_MFCC                RandomForest              30  0.6807 0.0333  0.6000  0.6000  -0.0807 ✓OK
  S2_MFCC                SVM                       30  0.6802 0.0339  0.5604  0.6000  -0.1197 ⚠OV
  S5_FusionEng           LogisticRegression        30  0.6720 0.0652  0.6267  0.6500  -0.0453 ✓OK
  S1_Spectrogram         SVM                       60  0.6675 0.1068  0.6491  0.6500  -0.0184 ✓OK
  S4_Fusion              RandomForest              50  0.6645 0.0390  0.6000  0.6000  -0.0645 ✓OK
  S3_Wav2Vec 

  Plots saved.

  CLASSIFICATION REPORTS — S1-S4 Best CV

  ── S1_Spectrogram × SVM (K=60) ──
  CV=0.6675±0.1068 | Test=0.6491 | Acc=0.6500
              precision    recall  f1-score   support

      Normal       0.67      0.60      0.63        10
     Depresi       0.64      0.70      0.67        10

    accuracy                           0.65        20
   macro avg       0.65      0.65      0.65        20
weighted avg       0.65      0.65      0.65        20


  ── S2_MFCC × RandomForest (K=30) ──
  CV=0.6807±0.0333 | Test=0.6000 | Acc=0.6000
              precision    recall  f1-score   support

      Normal       0.60      0.60      0.60        10
     Depresi       0.60      0.60      0.60        10

    accuracy                           0.60        20
   macro avg       0.60      0.60      0.60        20
weighted avg       0.60      0.60      0.60        20


  ── S3_Wav2Vec × LogisticRegression (K=70) ──
  CV=0.6891±0.0612 | Test=0.6491 | Acc=0.6500
              precision    